# Stage 2: Damage Segmentation (YOLOv8m-seg)

**Project:** Moroccan Car Damage Assessment — Pipeline Stage 2/3
**Author:** Hamza El Faghloumi (3A IATD-SI, ENSAM Meknès)

## Goal
Segment damage regions (dent, scratch, glass, broken_part) on car images. Will be combined with Stage 1 part masks at inference time.

## Dataset
**8-class Car Damages Roboflow dataset** → remapped to **4 classes**

## Class remapping policy
Original 8 classes → our 4 classes (decision rationale in comments):

| Original (Roboflow) | Our class | Rationale |
|---|---|---|
| dent | dent | direct |
| scratch | scratch | direct |
| crack | broken_part | structural damage, requires replacement |
| glass shatter | glass | windshield/window only |
| lamp broken | broken_part | headlight = part replacement (cher au Maroc) |
| tire flat | broken_part | tire/wheel replacement |
| dislocation | broken_part | misaligned panel |
| (any other) | broken_part | fallback |

**Note:** Verify the exact class names in your downloaded `data.yaml` and adjust the `REMAP` dict below.

## 1. Setup

In [ ]:
!pip install -q ultralytics==8.3.0 roboflow

import os, torch, yaml, shutil
from pathlib import Path
from ultralytics import YOLO
import ultralytics

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")
print(f"GPU:                 {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

## 2. Download the 8-class Car Damages dataset from Roboflow

**Get your API key:**
1. Go to roboflow.com → Settings → API Key
2. Store it as a Kaggle Secret named `ROBOFLOW_API_KEY` (Kaggle: Add-ons → Secrets)
3. The cell below loads it securely

**Choose your dataset:**
Find the exact workspace/project/version slug from the Roboflow dataset page URL.
Example URL: `https://universe.roboflow.com/<WORKSPACE>/<PROJECT>/dataset/<VERSION>`

In [ ]:
!pip install -q roboflow

In [ ]:
import roboflow
print("roboflow version:", roboflow.__version__)

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

RF_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')

# ⚠️ EDIT THESE for the specific 8-class dataset you chose
RF_WORKSPACE = 'martin-rpfil'
RF_PROJECT   = 'is_it_damaged'
RF_VERSION   = 6   # la plus récente "Accurate"

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8', location='/kaggle/working/damage_raw')

DAMAGE_RAW = Path(dataset.location)
print(f"Downloaded to: {DAMAGE_RAW}")
print("Original data.yaml:")
print((DAMAGE_RAW / 'data.yaml').read_text())

In [ ]:
# Vérifier si les annotations sont des bounding boxes ou des polygones
from pathlib import Path

lbl_dir = Path('/kaggle/working/damage_raw/train/labels')
sample_label = next(lbl_dir.glob('*.txt'))
print(f"Sample label: {sample_label.name}")
print("---")
print(sample_label.read_text()[:500])
print("---")

# Compte le nombre de valeurs par ligne
first_line = sample_label.read_text().strip().split('\n')[0]
n_values = len(first_line.split())
print(f"\nValeurs par ligne: {n_values}")
if n_values == 5:
    print("❌ Bounding box format (class x_center y_center width height) — PAS de segmentation!")
elif n_values > 5:
    print(f"✅ Polygon segmentation format (class + {n_values-1} polygon coordinates)")

## 3. Inspect original classes and define remapping

In [ ]:
# Load original class names
with open(DAMAGE_RAW / 'data.yaml') as f:
    orig_yaml = yaml.safe_load(f)

orig_names = orig_yaml['names']
if isinstance(orig_names, dict):
    orig_names = [orig_names[i] for i in sorted(orig_names.keys())]

print(f"Original {len(orig_names)} classes:")
for i, n in enumerate(orig_names):
    print(f"  {i}: {n}")

In [ ]:
# ⚠️ Mapping pour le dataset is_it_damaged (7 classes → 4 classes)
# La classe "background" sera supprimée (ce n'est pas un dommage).
#
# Nos 4 classes cibles:
#   0 = dent
#   1 = scratch
#   2 = glass        (windshield/window glass shatter only)
#   3 = broken_part  (crack, lamp, tire flat)

NAME_TO_NEW = {
    'dent':          0,
    'scratch':       1,
    'glass shatter': 2,
    'crack':         3,
    'lamp broken':   3,
    'tire flat':     3,
    'background':    None,   # à supprimer
}

# Build the index→index remap
REMAP = {}
SKIP_CLASSES = set()  # original indices to skip entirely
for old_idx, old_name in enumerate(orig_names):
    key = old_name.lower().strip()
    if key not in NAME_TO_NEW:
        raise ValueError(f"Class '{old_name}' (idx {old_idx}) not in NAME_TO_NEW. Add it.")
    new_idx = NAME_TO_NEW[key]
    if new_idx is None:
        SKIP_CLASSES.add(old_idx)
        print(f"⚠️  Will SKIP class {old_idx} ({old_name})")
    else:
        REMAP[old_idx] = new_idx

NEW_NAMES = {0: 'dent', 1: 'scratch', 2: 'glass', 3: 'broken_part'}

print("\nFinal remapping:")
for old_idx in sorted(set(range(len(orig_names))) - SKIP_CLASSES):
    new_idx = REMAP[old_idx]
    print(f"  {old_idx} ({orig_names[old_idx]:20s}) → {new_idx} ({NEW_NAMES[new_idx]})")
print(f"\nSkipped classes: {[orig_names[i] for i in SKIP_CLASSES]}")

## 4. Apply remapping to all label files

YOLO segmentation labels are `.txt` files with rows: `class_idx x1 y1 x2 y2 ...` (normalized polygon coords). We only rewrite the first token.

In [ ]:
# Copy raw dataset to a clean remapped version
DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')
if DAMAGE_REMAPPED.exists():
    shutil.rmtree(DAMAGE_REMAPPED)
shutil.copytree(DAMAGE_RAW, DAMAGE_REMAPPED)

# Find label dirs across train/valid/test
splits = []
for split_name in ['train', 'valid', 'test']:
    lbl_dir = DAMAGE_REMAPPED / split_name / 'labels'
    if lbl_dir.exists():
        splits.append((split_name, lbl_dir))

total_files = 0
total_lines_in = 0
total_lines_out = 0
remapped_lines = 0
skipped_lines = 0

for split_name, lbl_dir in splits:
    split_files = 0
    for txt in lbl_dir.glob('*.txt'):
        new_lines = []
        for line in txt.read_text().splitlines():
            if not line.strip():
                continue
            total_lines_in += 1
            parts = line.split()
            old_cls = int(parts[0])
            # Skip background annotations entirely
            if old_cls in SKIP_CLASSES:
                skipped_lines += 1
                continue
            new_cls = REMAP[old_cls]
            new_lines.append(' '.join([str(new_cls)] + parts[1:]))
            total_lines_out += 1
            if old_cls != new_cls:
                remapped_lines += 1
        # Write back (even if empty — YOLOv8 handles empty labels = negative example)
        txt.write_text('\n'.join(new_lines) + ('\n' if new_lines else ''))
        split_files += 1
        total_files += 1
    print(f"{split_name}: {split_files} files processed")

print(f"\nTotal files processed:     {total_files}")
print(f"Total annotation rows in:  {total_lines_in}")
print(f"Total annotation rows out: {total_lines_out}")
print(f"Rows where class changed:  {remapped_lines}")
print(f"Rows skipped (background): {skipped_lines}")

In [ ]:
from collections import Counter

print("Distribution after remapping:\n")
for split_name, lbl_dir in splits:
    counter = Counter()
    for txt in lbl_dir.glob('*.txt'):
        for line in txt.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    total = sum(counter.values())
    print(f"{split_name} (total: {total} annotations):")
    for cls_idx in sorted(NEW_NAMES.keys()):
        n = counter.get(cls_idx, 0)
        pct = (n/total*100) if total > 0 else 0
        print(f"  {cls_idx} {NEW_NAMES[cls_idx]:12s}: {n:5d}  ({pct:5.1f}%)")
    print()

## 5. Write new data.yaml

In [ ]:
data_yaml = {
    'path':  str(DAMAGE_REMAPPED),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'valid/images',
    'names': NEW_NAMES
}

yaml_path = Path('/kaggle/working/damage_data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(yaml_path.read_text())

## 6. Visualize remapped samples

In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np

def draw_yolo_polygons(img_path, lbl_path, class_names):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()

    colors = [(255,80,80), (80,180,255), (80,255,120), (255,200,60)]  # dent, scratch, glass, broken

    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split('\n'):
            if not line: continue
            parts = line.split()
            cls = int(parts[0])
            coords = list(map(float, parts[1:]))
            poly = np.array([[coords[i]*w, coords[i+1]*h] for i in range(0, len(coords), 2)], dtype=np.int32)
            cv2.fillPoly(overlay, [poly], colors[cls])
            x, y = poly[0]
            cv2.putText(img, class_names[cls], (x, max(y,15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

    return cv2.addWeighted(img, 0.55, overlay, 0.45, 0)

class_names = [NEW_NAMES[i] for i in sorted(NEW_NAMES.keys())]
imgs = list((DAMAGE_REMAPPED / 'train' / 'images').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_p in zip(axes.flat, imgs):
    lbl_p = DAMAGE_REMAPPED / 'train' / 'labels' / (img_p.stem + '.txt')
    ax.imshow(draw_yolo_polygons(img_p, lbl_p, class_names))
    ax.set_title(img_p.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Train YOLOv8m-seg on remapped 4-class dataset

**Strategy:**
- Same hyperparameters as Stage 1 for consistency.
- Class imbalance is expected (more scratches than glass) — `cls=0.5` keeps cls loss reasonable.
- Strong augmentation because damage detection is harder than part detection (less consistent appearance).

In [ ]:
!pip install -q --upgrade ultralytics

In [ ]:
import torch, numpy as np, ultralytics
from pathlib import Path
from ultralytics import YOLO

print(f"NumPy:       {np.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"GPU:         {torch.cuda.get_device_name(0)}")

# Vérifier que le dataset remappé est toujours là
DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')
yaml_path = Path('/kaggle/working/damage_data.yaml')
assert DAMAGE_REMAPPED.exists(), "Dataset remappé introuvable!"
assert yaml_path.exists(), "data.yaml introuvable!"
print(f"\n✅ Dataset prêt: {DAMAGE_REMAPPED}")
print(f"✅ Config YAML:  {yaml_path}")

In [ ]:
!pip install -q -U ultralytics numpy
!pip show ultralytics | grep Version
!pip show numpy | grep Version

In [ ]:
import torch, numpy as np, ultralytics, yaml
from pathlib import Path
from ultralytics import YOLO

print(f"NumPy:       {np.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"GPU:         {torch.cuda.get_device_name(0)}")

# Récupère les chemins (les fichiers sont déjà sur disque)
DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')
yaml_path = Path('/kaggle/working/damage_data.yaml')

assert DAMAGE_REMAPPED.exists(), "❌ Dataset perdu — il faut tout relancer"
assert yaml_path.exists(), "❌ YAML perdu — relance cellule 5"

print(f"\n✅ Dataset présent: {DAMAGE_REMAPPED}")
print(f"✅ YAML présent:    {yaml_path}")
print("\nContenu yaml:")
print(yaml_path.read_text())

In [ ]:
# Diagnostic : trouve les fichiers d'annotations malformés
from pathlib import Path

issues = []
DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')

for split in ['train', 'valid', 'test']:
    lbl_dir = DAMAGE_REMAPPED / split / 'labels'
    if not lbl_dir.exists():
        continue
    for txt in lbl_dir.glob('*.txt'):
        for line_num, line in enumerate(txt.read_text().splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            n = len(parts)
            # Une annotation segment doit avoir: class + paires (x,y) = 1 + 2k où k >= 3
            # Donc n-1 doit être pair ET n-1 >= 6
            n_coords = n - 1
            if n_coords < 6:
                issues.append((split, txt.name, line_num, f"trop court ({n_coords} coords)", line[:100]))
            elif n_coords % 2 != 0:
                issues.append((split, txt.name, line_num, f"nombre impair de coords ({n_coords})", line[:100]))
            elif n == 5:
                issues.append((split, txt.name, line_num, "format bbox (5 valeurs)", line))

print(f"Found {len(issues)} malformed annotation(s):\n")
for split, fname, line_num, reason, preview in issues[:20]:
    print(f"[{split}] {fname} line {line_num}: {reason}")
    print(f"   → {preview}")
    print()

In [ ]:
# Fix: supprime les lignes malformées + invalide les caches
import os
from pathlib import Path

DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')

cleaned_files = 0
removed_lines = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = DAMAGE_REMAPPED / split / 'labels'
    if not lbl_dir.exists():
        continue
    for txt in lbl_dir.glob('*.txt'):
        original_lines = txt.read_text().splitlines()
        good_lines = []
        for line in original_lines:
            if not line.strip():
                continue
            parts = line.split()
            n_coords = len(parts) - 1
            # Garde seulement les lignes avec polygone valide (>=6 coords, pair)
            if n_coords >= 6 and n_coords % 2 == 0:
                good_lines.append(line)
            else:
                removed_lines += 1
        if len(good_lines) != len([l for l in original_lines if l.strip()]):
            cleaned_files += 1
        txt.write_text('\n'.join(good_lines) + ('\n' if good_lines else ''))

print(f"✅ Cleaned {cleaned_files} files, removed {removed_lines} bad lines")

# CRITIQUE : supprimer le cache Ultralytics, sinon il utilisera l'ancienne version
for cache_file in ['train/labels.cache', 'valid/labels.cache', 'test/labels.cache']:
    cache_path = DAMAGE_REMAPPED / cache_file
    if cache_path.exists():
        os.remove(cache_path)
        print(f"   Deleted cache: {cache_path}")

print("\n🎯 Prêt pour le training")

In [ ]:
import torch, numpy as np, ultralytics, yaml
from pathlib import Path
from ultralytics import YOLO

print(f"NumPy:       {np.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"GPU:         {torch.cuda.get_device_name(0)}")
print(f"GPU mem free: {torch.cuda.mem_get_info(0)[0] / 1e9:.1f} GB / {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")

DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')
yaml_path = Path('/kaggle/working/damage_data.yaml')
assert DAMAGE_REMAPPED.exists() and yaml_path.exists(), "❌ Dataset manquant"
print(f"\n✅ Tout est prêt")

In [ ]:
!pip uninstall -y ultralytics
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=16,                # ← CHANGÉ de 32 à 16
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    close_mosaic=10,
    mixup=0.2,
    copy_paste=0.15,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    degrees=15.0,
    translate=0.15,
    scale=0.6,
    fliplr=0.5,
    cls=0.5,
    box=7.5,
    project='/kaggle/working/runs',
    name='damage_seg_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    device=0,
    workers=1,
    seed=42
)

In [ ]:
# Évaluation sur le test set (vraie performance finale)
best_weights = '/kaggle/working/runs/damage_seg_v1/weights/best.pt'
best_model = YOLO(best_weights)

metrics = best_model.val(
    data=str(yaml_path),
    split='test',
    imgsz=640,
    batch=16,
    plots=True
)

print(f"\nTest mAP50 (mask):   {metrics.seg.map50:.4f}")
print(f"Test mAP50-95 (mask): {metrics.seg.map:.4f}")
print(f"Test Precision:       {metrics.seg.mp:.4f}")
print(f"Test Recall:          {metrics.seg.mr:.4f}")

In [ ]:
# Sauvegarde finale du modèle dans un endroit propre
import shutil, os
final_path = '/kaggle/working/damage_segmenter_yolov8s.pt'
shutil.copy(best_weights, final_path)
print(f"✅ Sauvé : {final_path}")
print(f"   Taille : {os.path.getsize(final_path) / 1e6:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt
import cv2

test_imgs = list((DAMAGE_REMAPPED / 'test' / 'images').glob('*.jpg'))[:6]
preds = best_model.predict(test_imgs, imgsz=640, conf=0.25, save=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flat, preds):
    annotated = r.plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import os
from pathlib import Path
from datetime import datetime

# Liste les fichiers du dossier de training avec leur date de modification
runs_dir = Path('/kaggle/working/runs/damage_seg_v1')
if runs_dir.exists():
    print(f"Contenu de {runs_dir}:\n")
    for f in sorted(runs_dir.rglob('*'), key=lambda p: p.stat().st_mtime if p.exists() else 0):
        if f.is_file():
            mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%H:%M:%S')
            size_kb = f.stat().st_size / 1024
            print(f"  [{mtime}] {size_kb:8.1f} KB  {f.relative_to(runs_dir)}")
else:
    print("❌ Dossier de training introuvable")

In [ ]:
!nvidia-smi

## 8. Evaluate

In [ ]:
best_weights = '/kaggle/working/runs/damage_seg_v1/weights/best.pt'
best_model = YOLO(best_weights)

metrics = best_model.val(
    data=str(yaml_path),
    split='test' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'val',
    imgsz=640,
    batch=16,
    save_json=True,
    plots=True
)

print(f"\nTest mAP50 (box):  {metrics.box.map50:.4f}")
print(f"Test mAP50-95 (box): {metrics.box.map:.4f}")
print(f"Test mAP50 (mask): {metrics.seg.map50:.4f}")
print(f"Test mAP50-95 (mask): {metrics.seg.map:.4f}")

print("\nPer-class mask mAP50:")
for i, name in NEW_NAMES.items():
    print(f"  {name:15s}: {metrics.seg.maps[i]:.4f}")

## 9. Qualitative inference

In [ ]:
test_dir = DAMAGE_REMAPPED / ('test' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'valid') / 'images'
test_imgs = list(test_dir.glob('*.jpg'))[:4]

preds = best_model.predict(test_imgs, imgsz=640, conf=0.25, save=True,
                            project='/kaggle/working/runs', name='damage_inference', exist_ok=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, r in zip(axes, preds):
    annotated = r.plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Image as IPImage, display
from pathlib import Path

run_dir = Path('/kaggle/working/runs/damage_seg_v1')

print("📈 Courbes d'apprentissage (loss + mAP):")
display(IPImage(str(run_dir / 'results.png')))

print("\n📊 Matrice de confusion:")
display(IPImage(str(run_dir / 'confusion_matrix.png')))

print("\n📊 Matrice de confusion normalisée:")
display(IPImage(str(run_dir / 'confusion_matrix_normalized.png')))

## 10. Save final model

In [ ]:
final_path = '/kaggle/working/damage_segmenter_yolov8m.pt'
shutil.copy(best_weights, final_path)
print(f"✅ Final model saved to: {final_path}")
print(f"   Size: {os.path.getsize(final_path) / 1e6:.1f} MB")

In [ ]:
import shutil, os

best_weights = '/kaggle/working/runs/damage_seg_v1/weights/best.pt'
final_path = '/kaggle/working/damage_segmenter_yolov8s.pt'
shutil.copy(best_weights, final_path)
print(f"✅ Modèle final : {final_path}")
print(f"   Taille : {os.path.getsize(final_path) / 1e6:.1f} MB")

## Pipeline assembly preview

After both stages are trained, fusion looks like:

```python
parts = parts_model.predict(img)[0]      # 9 part masks
damages = damage_model.predict(img)[0]   # 4 damage masks

findings = []
for d_mask, d_cls in zip(damages.masks.data, damages.boxes.cls):
    for p_mask, p_cls in zip(parts.masks.data, parts.boxes.cls):
        overlap = (d_mask * p_mask).sum() / d_mask.sum()  # IoMin
        if overlap > 0.3:
            findings.append({
                'part': parts.names[int(p_cls)],
                'damage': damages.names[int(d_cls)],
                'overlap': float(overlap)
            })
```

Then plug `findings` into your Moroccan cost lookup table.